# 10-2 元組進階操作與可雜湊特性（Hashable）

- **單元編號**：10-2
- **學習目標**：
  1. 深入探究可變（Mutable）與不可變（Immutable）物件在記憶體中的運作本質與 `id()` 追蹤。
  2. 理解「可雜湊性（Hashability）」核心概念，洞悉為什麼只有不可變物件能作為雜湊鍵值。
  3. 掌握巢狀元組（Nested Tuples）與結構化多欄位資料的層級封裝與多層存取。
  4. 實作元組與方向向量的座標位移運算，建立幾何網格位移模型。
  5. 掌握多傳回值模擬機制，為後續模組化架構提早奠定穩固基礎。
  6. 精通元組比較大小（Tuple Comparison）與字典序規則，解鎖多條件排序的核心原型。
- **適合對象**：程式設計初學者（完全零基礎） / APCS 扎根學習者
- **先備知識**：10-1 元組基本建立與不可變性、第 9 章二維方向向量概念

---


### 10.2.1 可變（Mutable）與不可變（Immutable）物件的記憶體行為本質

在上一單元中，我們感受到元組「無法原地修改」的特性。但你有沒有想過：在電腦的記憶體底層，到底發生了什麼事？為什麼串列可以隨時改，而整數、浮點數、字串與元組卻不行？

#### 🔍 照妖鏡：內建函數 `id()`
在 Python 中，每一個物件在記憶體中都有一個獨一無二的門牌號碼，我們可以使用內建函數 **`id(obj)`** 查看它的記憶體位址！
- **可變物件（Mutable，如 List 串列）**：
  當我們執行 `my_list.append(99)` 或 `my_list[0] = 5` 時，`id(my_list)` **完全不會改變**！就像同一個白板，上面的字雖然換了，但白板本身還是掛在原來那間教室的同一個掛鉤上。
- **不可變物件（Immutable，如 int, str, tuple）**：
  當我們自以為在「修改」一個數值或字串時（例如 `x = 10; x = x + 1`），如果你查驗 `id(x)`，會震驚地發現門牌號碼完全變了！Python 是在記憶體中**重新開闢了一間新房間放 11**，然後把名牌 `x` 撕下來貼到新房間去！

#### ⚠️ 特殊邊界：元組內含可變串列的現象
如果一個元組內部裝了一個串列，例如 `t = ([1, 2], 3)`：
元組保護的是「**它手中握著的房間門牌號碼**」。元組記住了「第 0 格是一間串列房間、第 1 格是整數 3」，這兩個門牌永遠不能換！但是，串列房間內部的人員進出（`t[0].append(99)`），元組是管不著的！
因此在標準的軟體開發與 APCS 考場中，我們強烈建議：**元組內部應該全部存放不可變資料（如整數、字串、其他元組），打造絕對純淨堅固的安全實體！**


In [ ]:
# 範例 10.2.1：使用 id() 觀察可變串列 vs 不可變元組的記憶體行為

# 1. 串列（可變）：原地修改，記憶體位址 id 保持不變
demo_lst = [10, 20]
print("串列修改前 id：", id(demo_lst))
demo_lst.append(30)
demo_lst[0] = 99
print("串列修改後 id：", id(demo_lst), "（位址完全相同！）")

# 2. 元組（不可變）：任何修改嘗試都是產生全新物件
demo_t0 = (10, 20)
print()
print("元組原始 id：", id(demo_t0))
# 透過串接看似「加上新元素」，實則是建立了全新元組
demo_t0 = demo_t0 + (30,)
print("串接後的新元組 id：", id(demo_t0), "（位址已變為全新空間！）")

# 3. 探測元組內含串列的特殊現象
demo_hybr = ([1, 2], "Fixed")
print()
print("複合元組原始內容：", demo_hybr)
# 修改內含的串列（合法的，因為串列門牌沒變）
demo_hybr[0].append(3)
print("修改內含串列後的內容：", demo_hybr)

In [ ]:
# 填空 10.2.1：使用 id() 驗證記憶體位址變化
# 請將 ___ 替換為正確的函數名稱 id

base_t = (100, 200)
orgn_addr = ___(base_t)

# 重新賦值產生全新元組
base_t = (100, 200, 300)
new_addr = ___(base_t)

sm_objc = (orgn_addr == new_addr)
print("重新賦值前後記憶體位址是否相同：", sm_objc)

In [ ]:
# 練習 10.2.1：不可變字串與元組位址驗證儀
# 題目說明：輸入一個整數 X。
# 建立一個單元素元組 `t1 = (X,)`，記錄其 id 為 `id1`。
# 接著執行 `t2 = t1 + (X * 2,)`，記錄其 id 為 `id2`。
# 輸出兩行：
# 第一行：t2 的元素內容（以空白隔開）。
# 第二行：判斷 id1 是否等於 id2？若相同輸出 "SAME"，若不同輸出 "DIFFERENT"。

# 【公開測試資料 1】
# 5
# 輸出：
# 5 10
# DIFFERENT

# 【公開測試資料 2】
# 100
# 輸出：
# 100 200
# DIFFERENT

# 請在此處撰寫你的程式碼：


In [ ]:
# 挑戰 10.2.1：純淨元組檢測器
# 題目說明：給定一個容器列表。
# 檢驗該列表中所包含的各項元素是否「全為不可變的基本型態（int, float, str, tuple）」？
# 若全為不可變型態輸出 "SAFE"，若包含可變型態（如 list）輸出 "MUTABLE DETECTED"。
# 本題無公開測試資料，請自行測試。

# 請在此處撰寫你的程式碼：
test_items = [10, "Hello", (1, 2), 3.14]

is_safe = True
for item in test_items:
    if type(item) is list:
        is_safe = False
        break

if is_safe:
    print("SAFE")
else:
    print("MUTABLE DETECTED")


### 10.2.2 什麼是「可雜湊（Hashable）」？為什麼只有不可變物件能被雜湊？

我們常常聽到高階演算法中提到「**雜湊（Hash）**」、「雜湊表（Hash Table）」或是下一節即將學到的「字典（Dict）」與「集合（Set）」。這些以超光速 $O(1)$ 速度著稱的資料結構，背後仰賴的唯一基石就是——**物件必須是「可雜湊的（Hashable）」**！

#### 🔑 生活比喻：身分證指紋機
想像一間最高機密保險庫：
- 當你要存放一件物品時，管理員會把物品放到「**指紋掃描機（雜湊函數 Hash Function）**」上，計算出一串專屬的固定號碼（雜湊值 Hash Value），然後把物品存入該號碼對應的專屬儲物格中。
- 當你下次要找這件物品時，管理員不用從第 1 格翻到第 10000 格（$O(N)$ 線性搜尋），只要再掃描一次這件物品算出號碼，直接走向該格子，瞬間取出（$O(1)$ 查找）！

#### 💥 為什麼串列（List）絕對不能被雜湊？
現在想像一下：如果你把一個串列 `[1, 2]` 當作識別證放進儲物櫃。
結果過了一會兒，某人偷偷對這個串列執行了 `append(99)`，串列內容變成了 `[1, 2, 99]`！
當管理員再次掃描它時，算出來的號碼全變了，電腦根本找不到它原本被放在哪裡，整套系統立刻全面崩潰！
這就是為什麼 Python 規定：**只有內容一生一世絕不改變的不可變物件（如 int, str, tuple），才具備計算固定指紋的資格，稱為 Hashable！**

在 Python 中，你可以使用內建函數 **`hash(obj)`** 查驗物件的雜湊指紋：
- `hash((1, 2))` ➔ 算出正常整數指紋（合法！）
- `hash([1, 2])` ➔ 立即崩潰報錯：`TypeError: unhashable type: 'list'`（不可雜湊！）


In [ ]:
# 範例 10.2.2：使用 hash() 體驗可雜湊指紋與不可雜湊報錯

# 1. 不可變物件具備穩定的雜湊值（指紋）
demo_nhsh = hash(42)
demo_shsh = hash("APCS")
demo_thsh = hash((3, 5))

print("整數 42 的雜湊值：", demo_nhsh)
print("字串 'APCS' 的雜湊值：", demo_shsh)
print("座標元組 (3, 5) 的雜湊值：", demo_thsh)

# 2. 使用 try ... except 安全捕捉不可雜湊錯誤
demo_tcndd = [(1, 2), [1, 2], "Hello", 999]

print()
print("--- 雜湊資格檢驗表 ---")
for demo_candidate in demo_tcndd:
    try:
        demo_h = hash(demo_candidate)
        print(f"型態 {type(demo_candidate).__name__:>5}：合格！雜湊值 = {demo_h}")
    except TypeError:
        print(f"型態 {type(demo_candidate).__name__:>5}：❌ 違規！Unhashable 不可雜湊！")

In [ ]:
# 填空 10.2.2：檢查候選資料是否具備雜湊能力
# 請將 ___ 替換為正確的內建函數名稱 hash

coord = (10, 20)

# 計算不可變元組的雜湊指紋
h_val = ___(coord)

print("座標元組成功生成雜湊指紋：", h_val)


In [ ]:
# 練習 10.2.2：元組雜湊一致性驗證
# 題目說明：輸入兩個整數 A, B。
# 分別建立兩個獨立宣告的元組：
# `t1 = (A, B)` 與 `t2 = (A, B)`。
# 請計算兩者的 hash 值：`h1 = hash(t1)` 與 `h2 = hash(t2)`。
# 輸出兩行：
# 第一行：印出 t1 的雜湊值是否等於 t2 的雜湊值（若是輸出 "EQUAL"，若否輸出 "NOT EQUAL"）。
# 第二行：輸出 "A, B 可以安全作為字典的鍵值" 標籤提示。

# 【公開測試資料 1】
# 7 9
# 輸出：
# EQUAL
# HASHABLE OK

# 【公開測試資料 2】
# -3 100
# 輸出：
# EQUAL
# HASHABLE OK

# 請在此處撰寫你的程式碼：


In [ ]:
# 挑戰 10.2.2：動態資料不可變性守護檢測
# 題目說明：輸入一行包含 N 個以空白隔開的整數。
# 請將其封裝為元組 T。
# 請印出該元組的雜湊值是否為整數（isinstance(hash(T), int)）？
# 若成功印出 "PASSED"，若過程發生錯誤印出 "FAILED"。
# 本題無公開測試資料，請自行測試。

# 請在此處撰寫你的程式碼：
raw_vals = tuple(map(int, input().split()))
try:
    h = hash(raw_vals)
    if isinstance(h, int):
        print("PASSED")
    else:
        print("FAILED")
except:
    print("FAILED")


### 10.2.3 巢狀元組（Nested Tuples）與結構化多欄位資料打包（如 `(id, name, score)`）

在真實世界的程式設計與競賽中，資料往往不是單純的一串數字，而是具有層級與結構關聯的複合實體。例如一位選手的資料可能包含：
- 學號（整數）
- 姓名（字串）
- 多項成績（國、英、數三科成績組成的子元組）

#### 🪆 像俄羅斯套娃一樣嵌套：巢狀元組
元組內部可以容納任何型態，當然也包括「**另一個元組**」！這種元組裡面包元組的結構稱為 **巢狀元組（Nested Tuples）**。
```python
student = (101, "Alice", (95, 88, 92))
```
當我們要提取巢狀元組深處的資料時，使用**多重中括號索引**：
- `student[0]` ➔ 抓出學號 `101`
- `student[1]` ➔ 抓出姓名 `"Alice"`
- `student[2]` ➔ 抓出整包成績子元組 `(95, 88, 92)`
- `student[2][0]` ➔ 抓出成績子元組的第 0 項國文成績 `95`！

#### 💼 結構化優勢：免類別物件的超輕量封裝
在還沒學習物件導向（Class）之前，巢狀元組是組織複雜多欄位資料最乾淨、最不耗費額外記憶體、且天生不可竄改的極致利器！


In [ ]:
# 範例 10.2.3：巢狀元組的結構封裝與多層存取

demo_node = (1, (2, 4), "Checkpoint_A")

print("節點完整結構：", demo_node)
print("節點編號：", demo_node[0])
print("座標子元組：", demo_node[1])
print("列座標 r：", demo_node[1][0])
print("行座標 c：", demo_node[1][1])
print("節點名稱：", demo_node[2])

# 巢狀解包：可以直接拆解外層與內層！
demo_node_id, (demo_r, demo_c), demo_node_name = demo_node
print()
print(f"一鍵解包成果：ID={demo_node_id}, 位於第 {demo_r} 列第 {demo_c} 行，名稱為 {demo_node_name}")

In [ ]:
# 填空 10.2.3：多層巢狀解包提取學生數學成績
# 請將 ___ 替換為正確的變數名稱

rcrd102 = (101, "David", (88, 92, 95))

# 格式解包：學號, 姓名, (國, 英, 數)
sid, name, (chi, eng, ___) = rcrd102

print("姓名：", name)
print("數學成績：", math)


In [ ]:
# 練習 10.2.3：學生卡成績統計器
# 題目說明：輸入一行包含：
# 學號（整數）、姓名（字串）、三次段考成績（三個整數）。
# 請將其打包為巢狀元組：`(sid, nm, (score1, score2, score3))`。
# 請透過巢狀解包或多層索引計算三次成績的總分與整數平均（// 3）。
# 輸出兩行：
# 第一行：學號與姓名（以空白隔開）
# 第二行：總分與平均分（以空白隔開）

# 【公開測試資料 1】
# 101 Amy 80 90 100
# 輸出：
# 101 Amy
# 270 90

# 【公開測試資料 2】
# 205 Ken 70 75 80
# 輸出：
# 205 Ken
# 225 75

# 請在此處撰寫你的程式碼：
prts = input().split()
sid = int(prts[0])
nm = prts[1]
sc102 = (int(prts[2]), int(prts[3]), int(prts[4]))

stdn_crd = (sid, nm, sc102)

# 解包統計
s_id, s_name, (s1, s2, s3) = stdn_crd
tot102 = s1 + s2 + s3
avg102 = tot102 // 3

print(s_id, s_name)
print(tot102, avg102)


In [ ]:
# 挑戰 10.2.3：二維多線段長度統計
# 題目說明：輸入兩條線段，每行輸入四個整數：r1, c1, r2, c2 代表線段兩端點。
# 將每條線段儲存為巢狀元組：`((r1, c1), (r2, c2))`。
# 計算兩條線段的曼哈頓長度總和：(|r1 - r2| + |c1 - c2|)。
# 輸出兩線段長度之和。
# 本題無公開測試資料，請自行測試。

# 請在此處撰寫你的程式碼：
line1_pts = list(map(int, input().split()))
line2_pts = list(map(int, input().split()))

seg1 = ((line1_pts[0], line1_pts[1]), (line1_pts[2], line1_pts[3]))
seg2 = ((line2_pts[0], line2_pts[1]), (line2_pts[2], line2_pts[3]))

len1 = abs(seg1[0][0] - seg1[1][0]) + abs(seg1[0][1] - seg1[1][1])
len2 = abs(seg2[0][0] - seg2[1][0]) + abs(seg2[0][1] - seg2[1][1])

print(len1 + len2)


### 10.2.4 元組在二維座標位移運算中的應用（`(r + dr, c + dc)`）

在第九章 9-7 節中，我們學會了使用方向向量差值陣列 `dr = [-1, 1, 0, 0]` 與 `dc = [0, 0, -1, 1]` 來探測相鄰的四個格子。當時我們用兩個平行的數值變數進行步進：
```python
nr = r + dr[d]
nc = c + dc[d]
```
現在結合了元組，我們可以將這套二維網格導航系統升級為更簡潔、更直觀的**向量元組模型**！

#### 🧭 方向常數元組清單：
我們可以直接將四個方向定義為由四個「位移元組」組成的清單：
```python
DIRECTIONS = [
    (-1, 0),  # 0: 向上 (列 - 1, 行不變)
    (1, 0),   # 1: 向下 (列 + 1, 行不變)
    (0, -1),  # 2: 向左 (列不變, 行 - 1)
    (0, 1)    # 3: 向右 (列不變, 行 + 1)
]
```
當我們想要探測目前座標 `curr = (r, c)` 的四個相鄰鄰居時，迴圈寫法變得極具幾何美感：
```python
for dr, dc in DIRECTIONS:
    nr = r + dr
    nc = c + dc
    if 0 <= nr < R and 0 <= nc < C:
        # 合法鄰居座標 (nr, nc)
```
每個方向向量都是不可變的常數元組，不用擔心被迴圈意外覆蓋，這正是高分競技程式中最推崇的標準工程寫法！


In [ ]:
# 範例 10.2.4：方向向量元組與四方向相鄰座標探測實戰

demo_R, demo_C = 5, 5  # 5x5 網格
demo_spos = (2, 2)  # 中心出發點

# 定義四方向向量元組清單（上下左右）
demo_DRCT = [
    (-1, 0),  # 上
    (1, 0),   # 下
    (0, -1),  # 左
    (0, 1)    # 右
]

demo_r, demo_c = demo_spos
print(f"目前所在起點：({demo_r}, {demo_c})")

print()
print("--- 四方向相鄰合法鄰居座標 ---")
for demo_dr, demo_dc in demo_DRCT:
    demo_nr = demo_r + demo_dr
    demo_nc = demo_c + demo_dc
    # 邊界守護
    if 0 <= demo_nr < demo_R and 0 <= demo_nc < demo_C:
        demo_next_pos = (demo_nr, demo_nc)
        print(f"位移向量 ({demo_dr:+2d}, {demo_dc:+2d}) ➔ 到達鄰居：{demo_next_pos}")

In [ ]:
# 填空 10.2.4：補齊八方向相鄰探測之左上與右下對角向量
# 請將 ___ 替換為正確的元組數值

# 八方向向量清單（上下左右 + 四個對角線）
DRCT_8 = [
    (-1, 0), (1, 0), (0, -1), (0, 1),
    (-1, -1),  # 左上
    (-1, 1),   # 右上
    (1, -1),   # 左下
    (___, ___)    # 右下：列加 1，行加 1
]

print("八方向向量總數：", len(DRCT_8))


In [ ]:
# 練習 10.2.4：網格安全位移探測器
# 題目說明：輸入兩個整數 R, C 代表網格尺寸。
# 接著輸入起點座標 r, c 與方向代碼 D（0:上, 1:下, 2:左, 3:右）。
# 請使用方向向量元組清單計算移動後的新座標 (nr, nc)。
# 輸出規則：
# 若新座標仍在網格內（0 <= nr < R 且 0 <= nc < C），輸出 "VALID nr nc"。
# 若新座標撞牆出界，輸出 "OUT OF BOUNDS"。

# 【公開測試資料 1】
# 4 4
# 0 1 0
# 輸出：
# OUT OF BOUNDS

# 【公開測試資料 2】
# 4 4
# 1 1 3
# 輸出：
# VALID 1 2

# 請在此處撰寫你的程式碼：
R, C = map(int, input().split())
r, c, d_code = map(int, input().split())

DIRECTIONS = [(-1, 0), (1, 0), (0, -1), (0, 1)]

dr, dc = DIRECTIONS[d_code]
nr = r + dr
nc = c + dc

if 0 <= nr < R and 0 <= nc < C:
    print(f"VALID {nr} {nc}")
else:
    print("OUT OF BOUNDS")


In [ ]:
# 挑戰 10.2.4：連續向量步進路徑生成
# 題目說明：給定起點座標 (r, c)。
# 接著輸入一個整數 M 代表移動步數。
# 隨後有 M 行，每行輸入一個方向代碼 D（0:上, 1:下, 2:左, 3:右）。
# 請依序更新座標，並將過程中所踩過的「所有座標元組（包含起點）」加入一個串列中。
# 最後印出走過的所有座標（每行印一個元組）。
# 本題無公開測試資料，請自行測試。

# 請在此處撰寫你的程式碼：
r, c = map(int, input().split())
m102 = int(input())

DRCT = [(-1, 0), (1, 0), (0, -1), (0, 1)]
pth = [(r, c)]

for _ in range(m102):
    d = int(input())
    dr, dc = DRCT[d]
    r += dr
    c += dc
    pth.append((r, c))

for p in pth:
    print(p)


### 10.2.5 多傳回值模擬（為第 11 章自訂函式多值 return 提早鋪墊）

在後續的第十一章中，我們將會學習「自訂函數（Function）」。在許多其他程式語言（如 C/C++ 或 Java）中，一個函數通常只能嚴格回傳「單一一個數值」；如果想同時回傳「最大值」與「最小值」，往往要大費周章宣告指標或自訂結構。

但在 Python 中，工程師可以無比瀟灑地同時回傳多個值：
```python
# 預告第十一章的寫法：
# return min_val, max_val, total
```
#### 🎁 語法糖背後的功臣：自動元組打包（Tuple Packing）
為什麼 Python 能做到這件事？
因為當你用逗號隔開多個變數並輸出或賦值時，Python 在底層會**自動將它們打包為一個元組**！
就算你連小括號都懶得打：
```python
result = 10, 20, 30  # 這行完全合法！
print(type(result))  # 輸出：<class 'tuple'>
```
在主程式接收時，又透過上一節學過的「元組解包」，順暢地分派給各個變數：
```python
a, b, c = result
```
在我們當前尚未學習 `def` 函數的階段，這種「**打包多項統計指標為單一元組匯總物件**」的思維，能讓我們在處理多維統計資料時，邏輯極度清晰、程式模組化層次分明！


In [ ]:
# 範例 10.2.5：多值統計打包與解包接收實戰

demo_sc102 = [85, 92, 78, 96, 88]

demo_h102 = max(demo_sc102)
demo_l102 = min(demo_sc102)
demo_t102 = sum(demo_sc102)
demo_a102 = demo_t102 // len(demo_sc102)

demo_srprt = (demo_h102, demo_l102, demo_t102, demo_a102)

print("匯總報告元組：", demo_srprt)
print("報告型態：", type(demo_srprt))

# 接收端一鍵解包
demo_top, demo_bottom, demo_s_tot, demo_s_avg = demo_srprt
print(f"榜首分：{demo_top}，門檻分：{demo_bottom}，總合：{demo_s_tot}，均分：{demo_s_avg}")

In [ ]:
# 填空 10.2.5：將奇數與偶數總和打包為雙值元組
# 請將 ___ 替換為正確的變數

ns102 = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

odd_sum102 = sum(x for x in ns102 if x % 2 == 1)
evn_sum102 = sum(x for x in ns102 if x % 2 == 0)

# 打包成雙值元組
sums_tuple = (___, ___)

print("奇偶和元組：", sums_tuple)


In [ ]:
# 練習 10.2.5：四合一數據分析打包器
# 題目說明：輸入一串整數。
# 請計算出該數列的四項指標：
# 1. 元素總個數（N）
# 2. 最大值（MAX）
# 3. 最小值（MIN）
# 4. 總和（SUM）
# 將這四個數值依序打包為一個元組 `stats = (cnt, mx, mn, sm)`。
# 接著解包這四個數值，並輸出成單行（以空白隔開）。

# 【公開測試資料 1】
# 10 20 30 40
# 輸出：
# 4 40 10 100

# 【公開測試資料 2】
# 5 5 5
# 輸出：
# 3 5 5 15

# 請在此處撰寫你的程式碼：
ns7 = list(map(int, input().split()))

# 計算指標並打包
stats = (len(ns7), max(ns7), min(ns7), sum(ns7))

# 解包輸出
cnt, mx, mn, sm = stats
print(cnt, mx, mn, sm)


In [ ]:
# 挑戰 10.2.5：正負數與零計數三元組
# 題目說明：輸入一串整數。
# 統計其中：正數個數、負數個數、零的個數。
# 將三個計數打包為元組 `(pos_cnt, neg_cnt, zero_cnt)`。
# 印出該元組內容。
# 本題無公開測試資料，請自行測試。

# 請在此處撰寫你的程式碼：


### 10.2.6 元組比大小與字典序排序原則（多準則評比原型）

在第五章 5-1 節中，我們學過文字字串的比大小是依循「字典序（Lexicographical Order）」。那麼，如果拿「兩個元組」互相比大小，例如 `(a1, b1) < (a2, b2)`，Python 會怎麼判定勝負呢？

#### ⚖️ 元組比大小的黃金字典序原則：
元組比大小的邏輯非常清晰嚴謹，完全比照字典的查閱流程：
1. **先比第 0 個元素**：
   如果 `a1 < a2`，判定第一個元組較小，**立刻結束比對，完全不看後面的數字！**
   如果 `a1 > a2`，判定第一個元組較大，同樣立刻結束！
2. **第一項平手時（`a1 == a2`），才比第 1 個元素**：
   接著比對 `b1` 與 `b2`。依此類推，直到分出勝負為止。

#### 🏆 APCS 考場的制勝神器：多準則排序原型
這種特性在 APCS 考場上有著無比巨大的價值！
試想一個超經典考題：「**運動會選手排名**」：
- 首要條件：金牌數多的排前面。
- 次要條件：若金牌數相同，銀牌數多的排前面。
- 再次條件：若銀牌數依然相同，學號小的排前面！

在 Python 中，我們根本不需要寫複雜又容易 bug 的巢狀 if-else，只要為每位選手建構一個「**排序鍵值元組**」：
`(-gold, -silver, student_id)`
因為負金牌數由小到大排，就等於原金牌數由大到小排！Python 依據元組字典序比對，平手時自動往後遞補判定，一行就能實現極為高階的多條件排序！


In [ ]:
# 範例 10.2.6：元組比大小與多條件排序原型展示

demo_t1 = (5, 100)
demo_t2 = (8, 1)
print("(5, 100) < (8, 1) 結果：", demo_t1 < demo_t2)

demo_t3 = (5, 20)
demo_t4 = (5, 50)
print("(5, 20) < (5, 50) 結果：", demo_t3 < demo_t4)

demo_ath = [
    ("Alice", 2, 5, 101),
    ("Bob", 3, 1, 102),
    ("Charlie", 2, 5, 100),
    ("David", 2, 3, 104)
]

demo_rnkd = []
for demo_name, demo_g, demo_s, demo_sid in demo_ath:
    demo_rank_key = (-demo_g, -demo_s, demo_sid)
    demo_rnkd.append((demo_rank_key, demo_name, demo_g, demo_s, demo_sid))

demo_rnkd.sort()

print("排序後選手列表：")
for demo_item in demo_rnkd:
    print(demo_item[1], "成績元組：", demo_item[0])

In [ ]:
# 填空 10.2.6：建構雙條件元組進行大小比對
# 請將 ___ 替換為正確的變數，完成排序鍵值元組的建構

# 題目：比較兩位考生，優先比分數（高者勝），平手時比扣分次數（少者勝）
p1_score, p1_penalties = 90, 2
p2_score, p2_penalties = 90, 1

# 鍵值：(分數負號取反使高分排前, 扣分正數使少扣分者排前)
key1 = (-p1_score, p1_penalties)
key2 = (-p2_score, ___)

# 鍵值較小者代表表現更優秀（因為負分數更小代表原分數更高，罰分更少代表更小）
p2_bttr = (key2 < key1)
print("選手 2 是否勝出：", p2_bttr)


In [ ]:
# 練習 10.2.6：兩位選手多條件勝負判定
# 題目說明：輸入兩行，分別代表選手 A 與選手 B 的競賽數據。
# 每行三個整數：解題數（Solved）、總罰時（Penalty）、選手編號（ID）。
# 優勝規則：
# 1. 解題數較多者優勝。
# 2. 若解題數相同，總罰時較少者優勝。
# 3. 若依然相同，選手編號較小者優勝。
# 請將兩位選手的數據包裝為排序元組（利用負號進行方向切換），判定勝者並輸出該選手的 ID。

# 【公開測試資料 1】
# 4 120 1
# 5 300 2
# 輸出：
# 2

# 【公開測試資料 2】
# 4 150 10
# 4 120 20
# 輸出：
# 20

# 請在此處撰寫你的程式碼：
s1, p1, id1 = map(int, input().split())
s2, p2, id2 = map(int, input().split())

# 建構比較鍵值：(-solved, penalty, id)
ky1 = (-s1, p1, id1)
ky2 = (-s2, p2, id2)

if ky1 < ky2:
    print(id1)
else:
    print(id2)


In [ ]:
# 挑戰 10.2.6：三點距離原點多準則排序
# 題目說明：輸入三個座標點 r, c。
# 排序規則：
# 1. 距離原點曼哈頓距離（|r| + |c|）較小者排前面。
# 2. 若距離相同，r 座標較小者排前面。
# 3. 若 r 依然相同，c 座標較小者排前面。
# 請輸出排序後的第一名座標（以 r c 格式輸出）。
# 本題無公開測試資料，請自行測試。

# 請在此處撰寫你的程式碼：
pts = []
for _ in range(3):
    r, c = map(int, input().split())
    dist = abs(r) + abs(c)
    # 封裝元組：(距離, r, c)
    pts.append((dist, r, c))

pts.sort()
bst = pts[0]
print(bst[1], bst[2])


---
### 🧪【自主學習成效驗收】單元 10-2 一鍵自動評分與身分認證

恭喜你完成了單元 10-2 的所有探索與挑戰！🎉

剛才在各題中建立的數值型態變數、運算式與語法結構，現在都乖乖躺在記憶體置物櫃裡囉。  
點擊下方的 **▶ 播放鍵**，系統將跳出輸入框讓您登記**姓名**與**學號座號**（分開獨立欄位，亦可直接按 Enter 採用自學預設值），並自動啟動 Google 官方帳號身分認證（記錄真實 Email 與 Google 顯示姓名），產出**學習成果成績單**並同步登錄後台試算表！

> 💡 **貼心小叮嚀**：如果評分報告顯示某題未通過，別灰心！只要回到上方對應題目的儲存格修正程式碼，**重新點擊該題播放鍵 ▶ 執行**，再回來重新執行此評分儲存格就可以刷新成績與紀錄囉！

> 🆘 **終極救援方案**：如果重新執行個別題目後，成績還是怪怪的（例如明明寫對了卻還是顯示 ❌），最保險的做法是**整本全部重跑**，一鍵重置記憶體、告別幽靈變數！**Google Colab** 請點上方選單「**執行階段 ➔ 重新啟動工作階段並全部執行**」（快捷鍵 `Ctrl + F9`）；**VS Code** 請於上方工具列依序點選「**Clear All Outputs ➔ Restart ➔ Run All**」。

In [ ]:
#@title 🧪 點擊此處取得單元 10-2 學習成效自動評分報告 { display-mode: "form" }
# ------------------------------------------------------------------------------
# 官方智慧全環境評分載入器（支援：本地開發庫優先 ➔ 公開正式庫雙重備援）
# ------------------------------------------------------------------------------
import os
import sys
import json
import logging
import urllib.request

# 靜音內部 httplib2 警告
logging.getLogger("google_auth_httplib2").setLevel(logging.ERROR)

# 1. 互動式輸入學員姓名與學號（分開獨立欄位，按 Enter 採用預設值）
print("👤【學員身分登記】（自學測試可直接按 Enter 鍵使用預設值）：")
student_name = input("請輸入您的姓名 [預設: 自學冒險者]：").strip() or "自學冒險者"
student_id = input("請輸入學號或座號 [預設: 無座號]：").strip() or "無座號"

# 2. 自動嘗試獲取 Google 帳號官方身分認證（真實 Email 與 Google 暱稱）
google_email = None
google_name = None
try:
    from google.colab import auth
    print("🔐 正在連結 Google 帳號進行身分認證（若跳出授權視窗，請點選允許）...")
    auth.authenticate_user()

    # 方法 1: 使用 Google Drive API v3 (about:get) 抓取真實 displayName 與 emailAddress
    # Colab ADC 預設包含 drive 權限，能 100% 精準獲取 Google 帳號中文顯示名稱（如：葉明昌）
    try:
        import google.auth
        from googleapiclient.discovery import build
        creds, _ = google.auth.default()
        drive_service = build('drive', 'v3', credentials=creds, cache_discovery=False)
        about = drive_service.about().get(fields="user(displayName,emailAddress)").execute()
        u_obj = about.get("user", {})
        google_name = u_obj.get("displayName")
        google_email = u_obj.get("emailAddress")
    except Exception:
        pass

    # 方法 2: 若 Drive API 失敗，嘗試透過 OAuth2 userinfo / tokeninfo 備援
    if not google_email or not google_name:
        try:
            import google.auth
            from google.auth.transport.requests import Request
            credentials, _ = google.auth.default()
            credentials.refresh(Request())
            token = credentials.token

            for ep in [
                f"https://www.googleapis.com/oauth2/v3/tokeninfo?access_token={token}",
                "https://www.googleapis.com/oauth2/v3/userinfo",
                "https://www.googleapis.com/oauth2/v2/userinfo"
            ]:
                try:
                    req = urllib.request.Request(
                        ep,
                        headers={"Authorization": f"Bearer {token}"}
                    )
                    with urllib.request.urlopen(req, timeout=5) as resp:
                        u_data = json.loads(resp.read().decode('utf-8'))
                        if not google_email:
                            google_email = u_data.get("email") or u_data.get("emailAddress")
                        if not google_name:
                            google_name = (
                                u_data.get("name")
                                or u_data.get("given_name")
                                or u_data.get("displayName")
                            )
                        if google_email and google_name:
                            break
                except Exception:
                    continue
        except Exception:
            pass

    # 方法 3: 若只有 email 但 Google API 沒給 name，以 email 前綴作為暱稱回退
    if google_email and not google_name:
        google_name = google_email.split("@")[0]

    if google_email:
        print(f"✅ 身分認證成功！已綁定 Google 帳號：{google_name} ({google_email})")
except Exception:
    # 非 Colab 環境、本機離線測試或略過授權
    pass

# 3. 身分資料獨立注入環境變數（學號與姓名完全分開）
os.environ["STUDENT_NAME"] = student_name
os.environ["STUDENT_ID"] = student_id
if google_email:
    os.environ["GOOGLE_EMAIL"] = google_email
if google_name:
    os.environ["GOOGLE_NAME"] = google_name

# 4. 評分核心腳本載入（本機優先 ➔ 遠端記憶體載入，絕不產生實體暫存檔）
# 找不到來源就換下一條。內容已拿到但執行失敗時，停止改試其他來源。
# 評分腳本放在獨立命名空間執行，成功後也不把名稱寫回筆記本。答案讀執行前的學生存放格快照。
_loaded = False
_load_error = None

# 4.1 優先嘗試讀取開發庫/本機檔案（本機離線開發測試零延遲）
_local_paths = [
    "web/graders/grade_10_2.py",
    "../graders/grade_10_2.py",
    "../../graders/grade_10_2.py",
    "../web/graders/grade_10_2.py",
    "../../web/graders/grade_10_2.py"
]
for _lp in _local_paths:
    if not os.path.exists(_lp):
        continue
    try:
        with open(_lp, "r", encoding="utf-8") as _f:
            _code = _f.read()
        _grader_ns = {"__builtins__": __builtins__, "__name__": "apcs_grader", "__STUDENT_ENV__": dict(globals())}
        try:
            exec(_code, _grader_ns)
        finally:
            del _grader_ns
        os.environ["GRADER_SOURCE"] = _lp
        _loaded = True
        break
    except Exception as _exc:
        _load_error = _exc
        break

# 4.2 若在雲端環境（Colab），自動透過公開正式庫雙重備援以純記憶體載入
if not _loaded and _load_error is None:
    _grader_urls = [
        "https://raw.githubusercontent.com/johnnyy-lab/APCS1to3/main/graders/grade_10_2.py",
        "https://johnnyy-lab.github.io/APCS1to3/graders/grade_10_2.py",
        "https://cdn.jsdelivr.net/gh/johnnyy-lab/APCS1to3@main/graders/grade_10_2.py"
    ]
    for _url in _grader_urls:
        try:
            _req = urllib.request.Request(_url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(_req, timeout=10) as _resp:
                _code = _resp.read().decode("utf-8")
        except Exception:
            continue
        try:
            _grader_ns = {"__builtins__": __builtins__, "__name__": "apcs_grader", "__STUDENT_ENV__": dict(globals())}
            try:
                exec(_code, _grader_ns)
            finally:
                del _grader_ns
            os.environ["GRADER_SOURCE"] = _url
            _loaded = True
            break
        except Exception as _exc:
            _load_error = _exc
            break

if not _loaded:
    if _load_error is not None:
        print("❌ 評分程式載入失敗，請稍後重新執行。若仍失敗，請把這個畫面交給老師。")
        if os.environ.get("GRADER_DEBUG", "").strip().lower() in ("1", "true", "yes", "on"):
            _reason = str(_load_error).strip().splitlines()
            _line = _reason[0][:180] if _reason else ""
            if _line:
                print("（開發診斷：" + type(_load_error).__name__ + "：" + _line + "）")
            else:
                print("（開發診斷：" + type(_load_error).__name__ + "）")
    else:
        print("❌ 評分系統連線異常，請確認網路連線狀態（或稍後重新執行）")


## 本單元重點回顧與核心心法

在單元 10-2 中，我們從記憶體底層與演算法應用視角，全面解鎖了元組的高階威力：

1. **不可變性的記憶體本質**：
   - 串列修改時其 `id()` 門牌號碼保持不變，而元組內容一旦變更都是在記憶體中建立全新實體。
2. **可雜湊性（Hashability）指紋機制**：
   - 只有內容終生不變的不可變物件才能通過 `hash()` 計算出唯一指紋，這是元組能夠作為 `set` 元素與 `dict` 鍵值的底層鑰匙！
3. **巢狀元組結構化打包**：
   - 元組內嵌套元組，免類別物件即能實現多層級資料組織（如 `(id, name, (scores))`），支援多層索引與多層解包。
4. **二維向量位移模型**：
   - 將方向差值封裝為常數元組清單 `[(-1, 0), (1, 0), ...]`，大幅精簡網格走訪程式碼，兼具優雅與安全性。
5. **多傳回值與自動元組打包**：
   - 逗號隔開的多項數據會自動打包為元組，為跨區塊資料傳遞與多指標統計提供極致便利。
6. **元組字典序與多準則排序原型**：
   - 依序由左至右逐項對比，巧妙搭配負號（`-score`），一行代碼輕鬆實現 APCS 競賽中最繁瑣的多條件優先級排名！

---
🏆 **通關恭喜**！掌握了元組的不可變性與雜湊基礎後，你已經拿到了通往雜湊世界的黃金鑰匙！下一單元（10-3），我們將正式踏入 Python 最具殺傷力的王牌容器——**字典（Dictionary）**！
